In [66]:
import math
import random

import numpy as np
import matplotlib.pyplot as plt
from graphviz import Digraph
import networkx as nx
from pyvis.network import Network


random.seed(42)

In [67]:
# ---------- 计算图工具 ----------
def trace(root):
    nodes, edges = set(), set()

    def build(v):
        if v not in nodes:
            nodes.add(v)
            for parent in v._prev:
                edges.add((parent, v))
                build(parent)

    build(root)
    return nodes, edges

def draw_dot_nx_pyvis(root, out_html="ComputationGraph.html", open_browser=True):
    nodes, edges = trace(root)
    g = nx.DiGraph()

    # 1) 添加 Value 节点与 op 节点
    for n in nodes:
        # id(n) 返回对象的唯一内存地址（整数），用于生成唯一节点 ID
        val_id = f"val_{id(n)}"
        g.add_node(
            val_id,
            kind="value",
            label=f"{n.label} | data {n.data:.4f} | grad {n.grad:.4f}",
            shape="box",
            color="lightgrey",
        )

        # 如果该节点是由某个操作产生的，添加对应的操作节点
        if n._op:
            op_id = f"op_{id(n)}_{n._op}"
            g.add_node(
                op_id,
                kind="op",
                label=n._op,
                shape="circle",
                color="lightblue",
            )
            # 操作节点 -> 结果值节点
            g.add_edge(op_id, val_id)

    # 2) 添加 父值节点 -> 操作节点 的边
    for n1, n2 in edges:
        src = f"val_{id(n1)}"
        if n2._op:
            dst = f"op_{id(n2)}_{n2._op}"
            g.add_edge(src, dst)
        else:
            # 兜底：没有操作符时直接连到目标值节点
            dst = f"val_{id(n2)}"
            g.add_edge(src, dst)

    # 3) 用 pyvis 渲染为交互式 HTML
    net = Network(height="700px", width="100%", directed=True)
    net.from_nx(g)

    # 设置为类似 Graphviz 的从左到右层级布局
    net.set_options(
        """
        {
          "layout": {
            "hierarchical": {
              "enabled": true,
              "direction": "LR",
              "sortMethod": "directed",
              "levelSeparation": 160,
              "nodeSpacing": 140
            }
          },
          "physics": {"enabled": false},
          "edges": {
            "arrows": {"to": {"enabled": true}},
            "smooth": false
          },
          "nodes": {
            "font": {"size": 16}
          }
        }
        """
    )

    net.write_html(out_html, notebook=False, open_browser=open_browser)
    return g, net


In [68]:
# ---------- 自动微分核心 ----------
class Value:
    def __init__(self, data, _parent=(), _op='', label=''):
        # 当前节点保存的数值
        self.data = data

        # 当前节点在计算图中的“前驱节点”
        # 例如 z = x + y，则 z 的前驱就是 x 和 y
        self._prev = set(_parent)

        # 记录当前节点是通过什么运算得到的，便于调试/可视化
        self._op = _op

        # 节点标签，可选，用于调试或画图时显示名字
        self.label = label

        # 当前节点对应的梯度，初始为 0
        self.grad = 0.0

        # 反向传播函数：
        # 作用是在已知当前节点梯度的前提下，把梯度传给它的前驱节点
        self._backward = lambda: None

    def __repr__(self):
        # 打印对象时显示其数值
        return f"Value(data={self.data})"

    def __add__(self, other):
        # 支持 Value + 数字 的写法，数字自动包装成 Value
        other = other if isinstance(other, Value) else Value(other)

        # 前向计算：生成加法结果节点
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            # 若 out = self + other
            # 则 ∂out/∂self = 1, ∂out/∂other = 1
            # 根据链式法则：
            # self.grad += 1 * out.grad
            # other.grad += 1 * out.grad
            self.grad += out.grad
            other.grad += out.grad

        # 把当前运算对应的反向传播逻辑挂到结果节点上
        out._backward = _backward
        return out

    def __mul__(self, other):
        # 支持 Value * 数字
        other = other if isinstance(other, Value) else Value(other)

        # 前向计算：生成乘法结果节点
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            # 若 out = self * other
            # 则 ∂out/∂self = other.data
            #    ∂out/∂other = self.data
            # 根据链式法则继续往前传递梯度
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def __pow__(self, power):
        # 这里只支持 int / float 次幂
        assert isinstance(power, (int, float))

        # 前向计算：self 的 power 次方
        out = Value(self.data ** power, (self,), f'**{power}')

        def _backward():
            # 若 out = self ** power
            # 则 ∂out/∂self = power * self^(power-1)
            self.grad += power * (self.data ** (power - 1)) * out.grad

        out._backward = _backward
        return out

    def __neg__(self):
        # 取负号：-self 等价于 self * -1
        return self * -1

    def __sub__(self, other):
        # 减法：self - other 等价于 self + (-other)
        other = other if isinstance(other, Value) else Value(other)
        return self + (-other)

    def __truediv__(self, other):
        # 除法：self / other 等价于 self * (other ** -1)
        other = other if isinstance(other, Value) else Value(other)
        return self * (other ** -1)

    def __radd__(self, other):
        # 支持 数字 + Value
        return self + other

    def __rmul__(self, other):
        # 支持 数字 * Value
        return self * other

    def __rsub__(self, other):
        # 支持 数字 - Value
        return Value(other) + (-self)

    def tanh(self):
        # 前向计算：tanh 激活函数
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')

        def _backward():
            # tanh 的导数：
            # d/dx tanh(x) = 1 - tanh(x)^2
            # out.data 就是 tanh(self.data)
            self.grad += (1 - out.data ** 2) * out.grad

        out._backward = _backward
        return out

    def backward(self):
        # 反向传播主函数：
        # 目标是从当前节点出发，沿计算图反向计算所有相关节点的梯度

        topo = []
        visited = set()

        def build_topo(v):
            # 深度优先搜索，构建拓扑排序
            # 保证一个节点总是在其“后继节点”之前进入 topo
            if v not in visited:
                visited.add(v)
                for parent in v._prev:
                    build_topo(parent)
                topo.append(v)

        # 从当前输出节点开始构建整张依赖图的拓扑顺序
        build_topo(self)

        # 输出节点对自身的梯度恒为 1
        # 例如 y 对 y 的导数 dy/dy = 1
        self.grad = 1.0

        # 按拓扑序逆序遍历
        # 即从输出节点一路向前，把梯度逐步传回所有输入节点
        for node in reversed(topo):
            node._backward()

In [69]:
# ---------- 神经网络模块 ----------
class Neuron:  
    def __init__(self, nin):  
        # 初始化一个神经元：接收 nin 个输入（即输入特征维度）
        # 每个权重 w_i 是一个 Value 对象，用于支持自动微分
        # label 为可视化/调试时提供可读名称（如 'w0', 'w1', ...）
        self.w = [Value(random.uniform(-1, 1), label=f'w{i}') for i in range(nin)]  

        # 偏置项 b 也是一个 Value，参与自动微分
        self.b = Value(random.uniform(-1, 1), label='b')  

    def __call__(self, x):  
        # 前向传播：实现 y = tanh(∑ w_i * x_i + b)
        # x: 输入列表，每个元素是 Value 类型（例如 [x1, x2, ..., xn]）

        # 线性加权求和（含偏置）：
        # sum((wi * xi for wi, xi in zip(self.w, x)), self.b) 
        # 等价于：w0*x0 + w1*x1 + ... + wn*xn + b
        # 注意：Python 的 sum(start) 中 start 是初始值，这里传 self.b 作为累加起点
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)

        # 通过 tanh 激活函数非线性化
        out = act.tanh()
        # out.label = 'neuron_out'  # 为输出节点命名，便于后续可视化（如绘制计算图）

        return out  # 返回 Value 类型输出，可继续参与链式运算

    def parameters(self):  
        # 返回该神经元所有可训练参数（权重 + 偏置），全部为 Value 对象
        # 这些参数将在优化器中被更新（如 SGD、Adam）
        return self.w + [self.b]  

class Layer:  
    def __init__(self, nin, nout):  
        # 初始化一个全连接层：输入维度 nin，输出神经元数 nout
        # 即：该层有 nout 个并行的神经元，每个神经元有 nin 个输入权重
        self.neurons = [Neuron(nin) for _ in range(nout)]  

    def __call__(self, x):  
        # 前向传播：对每个神经元调用 __call__，输出一个列表
        # x 为输入向量（长度为 nin），每个神经元独立计算
        outs = [n(x) for n in self.neurons]

        # 为兼容单输出场景（如最后一层只有一个输出）：
        # 若仅一个神经元 → 返回单个 Value（而非列表）
        # 否则返回 list[Value]（如多分类输出）
        return outs[0] if len(outs) == 1 else outs  

    def parameters(self):  
        # 收集本层所有神经元的参数（展平为一维列表）
        # 格式：[w0_0, w0_1, ..., b0, w1_0, ..., b1, ...]
        return [p for n in self.neurons for p in n.parameters()]  

class MLP:  
    def __init__(self, nin, nouts):  
        # 初始化多层感知机（MLP）
        # nin: 输入特征维度
        # nouts: 各层输出神经元数量列表，如 [64, 32, 1] 表示 3 层（含输出层）
        sz = [nin] + nouts  # 拼接输入层 → 各隐藏层 → 输出层的维度序列
        # 例如 nin=3, nouts=[4,2] ⇒ sz = [3, 4, 2] ⇒ 构建两层：Layer(3,4) 和 Layer(4,2)
        self.layers = [Layer(sz[i], sz[i + 1]) for i in range(len(nouts))]

    def __call__(self, x):  
        # 前向传播主流程：
        # 输入 x 被逐层传递，每层输出作为下一层输入
        # 注意：x 可能是单个 Value（标量）或 list[Value]（向量）
        for layer in self.layers:
            x = layer(x)  # layer(x) 返回单个 Value 或 list[Value]
        return x  # 最终输出（标量或向量）

    def parameters(self):  
        # 收集整个网络的所有可训练参数（所有权重与偏置）
        # 用于后续反向传播 + 优化器更新
        return [p for layer in self.layers for p in layer.parameters()]

## 1) Value 示例
目标：理解最基础的标量计算图和梯度传播。

In [71]:
# 前向：loss = ((a*b + c) - 10)^2
a = Value(2.0, label='a')   # a = 2
b = Value(-3.0, label='b')  # b = -3
c = Value(10.0, label='c')  # c = 10

d = a * b
d.label = 'd'               # d = -6

e = d + c
e.label = 'e'               # e = 4

f = e - 10.0
f.label = 'f'               # f = -6

loss_v = f ** 2
loss_v.label = 'loss_v'     # loss_v = 36

# 反向
loss_v.backward()

print('Value 示例:')
print(f'loss={loss_v.data:.4f}, da={a.grad:.4f}, db={b.grad:.4f}, dc={c.grad:.4f}')
draw_dot_nx_pyvis(loss_v)

Value 示例:
loss=36.0000, da=36.0000, db=-24.0000, dc=-12.0000


(<networkx.classes.digraph.DiGraph at 0x17b7ff10e50>,
 <class 'pyvis.network.Network'> |N|=15 |E|=14)

## 2) Neuron 示例
目标：看一个神经元如何把多个输入变成一个输出。

In [72]:
neu = Neuron(3)
x = [2.0, 3.0, -1.0]   # x = [2, 3, -1]. 单个样本，3个特征
target = 1.0           # target = 1

pred_n = neu(x)
pred_n.label = 'pred_n'   # pred_n = 神经元输出

diff_n = pred_n - target
diff_n.label = 'diff_n'   # diff_n = pred_n - 1

loss_n = diff_n ** 2
loss_n.label = 'loss_n'   # loss_n = diff_n^2

for p in neu.parameters():
    p.grad = 0.0          # 所有参数梯度清零

loss_n.backward()         # 反向传播，计算各参数梯度

print('Neuron 示例:')
print(f'pred={pred_n.data:.4f}, loss={loss_n.data:.4f}')
print(f'w0.grad={neu.w[0].grad:.4f}, b.grad={neu.b.grad:.4f}')
draw_dot_nx_pyvis(loss_n)

Neuron 示例:
pred=-0.9835, loss=3.9344
w0.grad=-0.2590, b.grad=-0.1295


(<networkx.classes.digraph.DiGraph at 0x17b7ff05e50>,
 <class 'pyvis.network.Network'> |N|=29 |E|=28)

## 3) Layer 示例
目标：看一层多个神经元如何并行产生多个输出。

In [73]:
layer = Layer(3, 2)         # layer = 1个输入3维、输出2维的层
x = [1.0, -2.0, 0.5]        # x = [1, -2, 0.5]
target0 = 1.0               # target0 = 1
target1 = -1.0              # target1 = -1

outs_l = layer(x)           
# outs_l = [o0, o1]

o0 = outs_l[0]
o0.label = 'o0'             # label: o0，o0 = 第1个输出值

o1 = outs_l[1]
o1.label = 'o1'             # label: o1，o1 = 第2个输出值

d0 = o0 - target0
d0.label = 'd0'             # label: d0，d0 = o0 - target0

d1 = o1 - target1
d1.label = 'd1'             # label: d1，d1 = o1 - target1

s0 = d0 ** 2
s0.label = 's0'             # label: s0，s0 = d0^2

s1 = d1 ** 2
s1.label = 's1'             # label: s1，s1 = d1^2

loss_l = s0 + s1
loss_l.label = 'loss_l'     # label: loss_l，loss_l = s0 + s1

for p in layer.parameters():
    p.grad = 0.0            # 每个参数的 grad = 0

loss_l.backward()           # 反向传播，计算所有参数梯度

print('Layer 示例:')
print(f'o0={o0.data:.4f}, o1={o1.data:.4f}, loss={loss_l.data:.4f}')
print(f'参数总数={len(layer.parameters())}, 第1个参数梯度={layer.parameters()[0].grad:.4f}')
draw_dot_nx_pyvis(loss_l)

Layer 示例:
o0=-0.5835, o1=0.8965, loss=6.1042
参数总数=8, 第1个参数梯度=-2.0887


(<networkx.classes.digraph.DiGraph at 0x17b7f9868b0>,
 <class 'pyvis.network.Network'> |N|=60 |E|=59)

## 4) MLP 示例
目标：看多层网络从输入到输出、再到 loss 的完整自动微分流程。

In [74]:
mlp = MLP(3, [4, 4, 1])     # mlp = 输入3维，隐藏层4/4，输出1维
x = [2.0, 3.0, -1.0]        # x = [2, 3, -1]
target = 1.0                # target = 1

pred_m = mlp(x)
pred_m.label = 'pred_m'     # label: pred_m，pred_m = MLP 输出值

diff_m = pred_m - target
diff_m.label = 'diff_m'     # label: diff_m，diff_m = pred_m - target

loss_m = diff_m ** 2
loss_m.label = 'loss_m'     # label: loss_m，loss_m = diff_m^2

for p in mlp.parameters():
    p.grad = 0.0            # 每个参数的 grad = 0

loss_m.backward()           # 反向传播，计算所有参数梯度

lr = 0.05                   # 学习率 = 0.05
for p in mlp.parameters():
    p.data -= lr * p.grad   # 参数更新：p = p - lr * grad

print('MLP 示例:')
print(f'pred={pred_m.data:.4f}, loss={loss_m.data:.4f}, 参数数量={len(mlp.parameters())}')
print(f'示例更新后首个参数={mlp.parameters()[0].data:.4f}')
draw_dot_nx_pyvis(loss_m)

MLP 示例:
pred=-0.0684, loss=1.1416, 参数数量=41
示例更新后首个参数=-0.9469


(<networkx.classes.digraph.DiGraph at 0x17b01b91ca0>,
 <class 'pyvis.network.Network'> |N|=207 |E|=218)

### 小结
- `Value`：最小单元，负责记录数据、梯度和局部反传逻辑。
- `Neuron`：把多个 `Value` 组合成一个非线性单元。
- `Layer`：多个神经元并行。
- `MLP`：多层串联，形成完整网络。

建议：按顺序运行每个单元，观察每次计算图中 `grad` 的变化。